# SO-101: ghi dữ liệu, train ACT, đánh giá trên Google Colab

Notebook này làm đúng 3 bước như trên máy ở nhà (xem `docs/so_tay_ai.md`), chỉ khác là chạy trên GPU của Colab:

1. **Ghi dữ liệu**: policy viết tay chạy trong MuJoCo, ghi ảnh + góc khớp + lệnh (hoặc dùng dataset đã tải lên Drive).
2. **Train** ACT bằng `lerobot-train`.
3. **Đánh giá** trên bộ seed cố định, ghi replay để xem trên Studio ở máy nhà.

Kết quả lưu vào Google Drive (`MyDrive/vla/...`) để không mất khi Colab ngắt.

**Trước khi chạy:** menu *Runtime → Change runtime type → GPU* (Pro: chọn **A100** hoặc **L4**; T4 cũng được nhưng chậm hơn).
Chạy lần lượt từng ô từ trên xuống (Shift+Enter).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Nối Google Drive

Mọi thứ cần giữ lại (dataset, checkpoint, kết quả) nằm trong `MyDrive/vla`. Colab sẽ hỏi quyền truy cập Drive: bấm cho phép.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
VLA = '/content/drive/MyDrive/vla'          # thư mục gốc trên Drive, giống D:/vla ở máy nhà
for sub in ('datasets', 'outputs/train', 'eval', 'replays'):
    os.makedirs(f'{VLA}/{sub}', exist_ok=True)
print('Drive sẵn sàng:', VLA)

## 3. Lấy code và cài thư viện

Code lấy từ GitHub (nhánh `my-work`). Thư mục phải tên `sim_engine` vì Python import theo tên đó.
`MUJOCO_GL=egl`: Colab không có màn hình, MuJoCo vẽ ảnh camera bằng GPU qua EGL.
Cài mất khoảng 3-5 phút. Nếu Colab báo *Restart session* sau khi cài thì bấm restart rồi chạy lại từ ô số 2.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['HF_HOME'] = '/content/hf-cache'
# Khai báo driver EGL của NVIDIA: thiếu file này MuJoCo vẽ ảnh bằng CPU (llvmpipe), chậm ~160 lần (0,85 s mỗi ảnh)
p = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(p):
    open(p, 'w').write('{"file_format_version" : "1.0.0", "ICD" : {"library_path" : "libEGL_nvidia.so.0"}}')
if os.path.isdir('/content/sim_engine/.git'):
    !git -C /content/sim_engine pull -q
else:
    !git clone -q -b my-work https://github.com/tvhhai/sim-engine.git /content/sim_engine
!git -C /content/sim_engine log --oneline -1
!pip install -q "lerobot[dataset,training]==0.7.0" "mujoco>=4.0.1" gymnasium opencv-python-headless h5py

In [ ]:
# Kiểm tra nhanh: dựng env, vẽ một ảnh camera
%cd /content
import os
os.environ['MUJOCO_GL'] = 'egl'
from sim_engine.learning import sim_setup
env, opt = sim_setup.make_env(3000)
obs, _ = env.reset(seed=3000, options=opt)
frame = sim_setup.to_frame(env, obs, [0] * 7, 'test', ('top', 'wrist'))
print({k: getattr(v, 'shape', v) for k, v in frame.items()})
env.close()

## 4. Chọn thí nghiệm

Mỗi thí nghiệm là một file trong `learning/configs/`. Đổi `EXP` và các thông số dưới đây cho khớp:

| EXP | CAMERAS | ENV_STATE | NOISE |
|---|---|---|---|
| `act_box4_v1` | `front,wrist` | False | 0 |
| `act_box4_topwrist` | `top,wrist` | False | 0 |
| `act_box4_wrist` | `wrist` | False | 0 |
| `act_box4_state` | (để trống) | True | 0 |
| `act_box4_topwrist_noise` | `top,wrist` | False | 0.02 |

`BATCH`: GPU Colab lớn hơn GTX 1650 nên tăng được (A100: 32-64, L4: 16-32, T4: 8-16). Đổi batch thì kết quả không còn so trực tiếp
với bản train ở nhà (batch 8); muốn so công bằng thì để 8.

In [ ]:
EXP = 'act_box4_topwrist'      # tên file config (không có .yaml)
CAMERAS = 'top,wrist'
ENV_STATE = False
NOISE = 0.0
EPISODES = 200                 # số ván thành công cần ghi
BATCH = 32
STEPS = 20000

DS_NAME = 'so101_box4_' + EXP.replace('act_box4_', '')        # vd so101_box4_topwrist
DS_LOCAL = f'/content/datasets/{DS_NAME}'                      # dataset đọc từ ổ của máy Colab (nhanh hơn đọc từ Drive)
DS_ZIP = f'{VLA}/datasets/{DS_NAME}.zip'                       # bản lưu trên Drive
OUT = f'{VLA}/outputs/train/{EXP}_colab_b{BATCH}'              # checkpoint lưu thẳng lên Drive
print(DS_LOCAL, DS_ZIP, OUT, sep='\n')

## 5. Có dataset

Ô dưới tự chọn:
- Đã có `MyDrive/vla/datasets/<tên>.zip` (ghi ở lần trước, hoặc bạn tải từ máy nhà lên) thì giải nén ra dùng.
- Chưa có thì ghi mới trên Colab (200 ván, khoảng 20-40 phút tuỳ máy), rồi nén lưu lên Drive cho lần sau.

Tải dataset từ máy nhà lên (không bắt buộc): nén thư mục `D:/vla/datasets/<tên>` thành `<tên>.zip`, kéo vào `MyDrive/vla/datasets/` trên drive.google.com.
Zip phải chứa thẳng các thư mục `data/ meta/ videos/` (không lồng thêm thư mục).

In [ ]:
import os, shutil, subprocess
if os.path.isdir(DS_LOCAL):
    print('Đã có dataset trên máy Colab:', DS_LOCAL)
elif os.path.exists(DS_ZIP):
    os.makedirs(DS_LOCAL, exist_ok=True)
    !unzip -q "{DS_ZIP}" -d "{DS_LOCAL}"
    print('Đã giải nén từ Drive:', DS_ZIP)
else:
    extra = (' --env-state' if ENV_STATE else '') + (f' --noise {NOISE}' if NOISE else '')
    !cd /content && python -m sim_engine.learning.record_dataset --episodes {EPISODES} --cameras "{CAMERAS}" {extra} --root {DS_LOCAL} --repo-id local/{DS_NAME} 2>&1 | grep -E "seed|Xong" | awk 'NR % 20 == 0 || /Xong/'
    shutil.make_archive(DS_ZIP[:-4], 'zip', DS_LOCAL)
    print('Đã ghi và lưu lên Drive:', DS_ZIP)
!du -sh {DS_LOCAL}

## 6. Train

In loss mỗi 100 bước; checkpoint mỗi 5000 bước vào `OUT` trên Drive.
Nếu Colab ngắt giữa chừng: chạy lại ô 2, 3, 4, 5 rồi chạy ô **6b (train tiếp)** thay vì ô này.

In [ ]:
!cd /content && python -m sim_engine.learning.train sim_engine/learning/configs/{EXP}.yaml --dataset.root={DS_LOCAL} --output_dir={OUT} --batch_size={BATCH} --steps={STEPS} --num_workers=4 2>&1 | grep -E --line-buffered "step:|Error|Traceback|error"

### 6b. Train tiếp sau khi bị ngắt

In [ ]:
!cd /content && lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true --dataset.root={DS_LOCAL} 2>&1 | grep -E --line-buffered "step:|Error|Traceback|error"

## 7. Đánh giá + replay

Chạy 100 seed của bộ đánh giá (giống ở nhà). Replay chép lên `MyDrive/vla/replays/`: tải file `.json` về đặt vào
`sim-engine/results/replays/` ở máy nhà, mở Studio → Replay → Nguồn replay.

In [ ]:
POLICY = f'{OUT}/checkpoints/last/pretrained_model'
NAME = f'{EXP}_colab'
!cd /content && python -m sim_engine.learning.eval_policy --policy {POLICY} --limit 100 --name {NAME} --out {VLA}/eval/{NAME}.json --replay {NAME} 2>&1 | grep -vE "Warning|warn"
!cp /content/sim_engine/results/replays/{NAME}.json {VLA}/replays/
print('Replay:', f'{VLA}/replays/{NAME}.json')

## 9. Cách khác: chạy cả bước 5-6-7 ở nền (để qua đêm)

Ô 5, 6, 7 chạy trong trang web: trang mất kết nối là hàng chờ ô bị mất. Ô dưới viết các bước đó thành một script và chạy **ở nền
trên máy Colab** (`nohup`), nên trang có mất kết nối thì máy vẫn làm tiếp. Chạy ô 1-4 trước (GPU, Drive, cài đặt, chọn thí nghiệm),
rồi chạy ô này **thay cho** ô 5-6-7. Xem tiến độ bằng ô kế tiếp (chạy lại bao nhiêu lần cũng được).

Lưu ý: Colab Pro vẫn tắt máy nếu không còn tab nào mở notebook quá lâu; cứ để tab mở.

In [ ]:
extra = (' --env-state' if ENV_STATE else '') + (f' --noise {NOISE}' if NOISE else '')
script = f'''set -x
export MUJOCO_GL=egl HF_HOME=/content/hf-cache
cd /content
if [ ! -d {DS_LOCAL} ]; then
  if [ -f {DS_ZIP} ]; then mkdir -p {DS_LOCAL} && unzip -q {DS_ZIP} -d {DS_LOCAL}
  else python -m sim_engine.learning.record_dataset --episodes {EPISODES} --cameras "{CAMERAS}"{extra} --root {DS_LOCAL} --repo-id local/{DS_NAME} && (cd {DS_LOCAL} && zip -qr {DS_ZIP} .)
  fi
fi
python -m sim_engine.learning.train sim_engine/learning/configs/{EXP}.yaml --dataset.root={DS_LOCAL} --output_dir={OUT} --batch_size={BATCH} --steps={STEPS} --num_workers=4
python -m sim_engine.learning.eval_policy --policy {OUT}/checkpoints/last/pretrained_model --limit 100 --name {EXP}_colab --out {VLA}/eval/{EXP}_colab.json --replay {EXP}_colab
cp /content/sim_engine/results/replays/{EXP}_colab.json {VLA}/replays/
echo ALL_DONE
'''
open('/content/run_all.sh', 'w').write(script)
!nohup bash /content/run_all.sh > /content/run.log 2>&1 &
print('Đã chạy nền. Xem tiến độ bằng ô dưới.')

In [ ]:
# Tiến độ: ghi dữ liệu (seed ...), train (step: ...), chấm điểm ([k/100] ...), xong hẳn khi thấy ALL_DONE
!tr '\r' '\n' < /content/run.log | grep -E "seed [0-9]+:|Xong|step:|\[[0-9]+/100\]|\"all\"|ALL_DONE|Error|Traceback" | tail -n 6
!cp /content/run.log {VLA}/{EXP}_colab_run.log

## 10. DAgger: AI lái, thầy sửa, train tiếp (chạy nền)

Dùng sau khi đã có một mạng train xong (vd `act_box4_topwrist_500`, 59/100). Chạy ô 1-4 trước (GPU, Drive, cài đặt; ô 4 chỉ để có
biến `VLA`), rồi chạy ô dưới. Các bước, tất cả ở nền như ô 9:

1. **Ghi DAgger** (`learning/dagger.py`): mỗi ván AI tự lái một đoạn ngẫu nhiên (15-250 bước), rồi thầy giáo (policy viết tay) tiếp quản
   từ đúng chỗ đó và làm nốt. Ghi đoạn thầy làm. Ván AI tự xong thì bỏ (không có gì để sửa). Ghi theo **từng phần `CHUNK` ván**, xong
   phần nào zip lên Drive phần đó.
2. **Gộp** dataset gốc + các phần DAgger (`learning/merge_datasets.py`).
3. **Train tiếp** mạng cũ trên dataset gộp `FT_STEPS` bước (`learning/configs/act_box4_topwrist_500_dagger1.yaml`).
4. **Chấm** 100 seed, chép replay lên Drive.

**Colab thu hồi máy giữa chừng** (mất `/content`): chạy lại ô 1-4 rồi ô dưới. Phần DAgger đã có zip trên Drive thì bỏ qua (chỉ mất phần
đang ghi dở), train có checkpoint thì chạy tiếp từ checkpoint (mất tối đa 5000 bước), đã chấm xong thì không chấm lại.

Vòng sau: tăng `ROUND` (vòng 2, 3...). `BASE_OUT` tự lấy mạng của vòng trước; dataset gộp = gốc + mọi vòng DAgger (zip trên Drive).

In [ ]:
ROUND = 3
# mạng hiện tại: AI lái khi ghi DAgger, và điểm xuất phát để train tiếp (vòng 1: mạng 500 ván; vòng sau: mạng của vòng trước)
BASE_OUT = (f'{VLA}/outputs/train/act_box4_topwrist_500_colab_l4_b32' if ROUND == 1
            else f'{VLA}/outputs/train/act_box4_topwrist_500_dagger{ROUND - 1}_colab')
BASE_CKPT = '010000'                                                       # checkpoint của BASE_OUT dùng làm điểm xuất phát ('last' hoặc '010000'...): vòng 2 last 68 %, 010000 85 %
BASE_DS = 'so101_box4_topwrist_500'                                        # dataset gốc (zip trên Drive)
CHUNK, PARTS = 50, 10                                                      # ghi 10 phần x 50 ván thầy sửa = 500 ván (vòng 1-2: 6 phần)
FT_STEPS = 30000                                                           # vòng 1-2: 20000; checkpoint tốt nhất thường ở giữa nên chấm mọi checkpoint
DAG = lambda r: f'so101_box4_topwrist_dagger{r}'
DAG_DS = DAG(ROUND)
MERGED = f'so101_box4_topwrist_500_dagger{ROUND}'
FT_OUT = f'{VLA}/outputs/train/act_box4_topwrist_500_dagger{ROUND}_colab'
NAME = f'act_box4_topwrist_500_dagger{ROUND}_colab'
SEED0 = 6000 + (ROUND - 1) * 10000                                         # mỗi phần dùng dải seed riêng (1000 seed), mỗi vòng dải riêng
# gộp dataset gốc + MỌI vòng DAgger tới giờ (vòng trước lấy từ zip trên Drive; mỗi vòng có thể có số phần khác nhau nên dùng glob)
parts = ' '.join(f'/content/datasets/{DAG(r)}_p[0-9]*' for r in range(1, ROUND + 1))
lines = [
    'set -x',
    'export MUJOCO_GL=egl HF_HOME=/content/hf-cache',
    'cd /content && mkdir -p /content/datasets',
    f'if [ ! -d /content/datasets/{BASE_DS} ]; then mkdir -p /content/datasets/{BASE_DS} && unzip -q {VLA}/datasets/{BASE_DS}.zip -d /content/datasets/{BASE_DS}; fi',
    *[f'for Z in {VLA}/datasets/{DAG(r)}_p[0-9]*.zip; do P=/content/datasets/$(basename $Z .zip); [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P) || exit 1; done'
      for r in range(1, ROUND)],
    f'for K in $(seq 0 {PARTS - 1}); do',
    f'  P=/content/datasets/{DAG_DS}_p$K; Z={VLA}/datasets/{DAG_DS}_p$K.zip',
    '  if [ -d $P ] && [ -f $Z ]; then continue; fi',
    '  rm -rf $P',
    '  if [ -f $Z ]; then mkdir -p $P && unzip -q $Z -d $P',
    f'  else python -m sim_engine.learning.dagger --policy {BASE_OUT}/checkpoints/{BASE_CKPT}/pretrained_model --episodes {CHUNK} --seed-start $(({SEED0} + K * 1000)) --root $P --repo-id local/{DAG_DS}_p$K || exit 1',
    '    (cd $P && zip -qr /content/part.zip . && cp /content/part.zip $Z.tmp && mv $Z.tmp $Z && rm /content/part.zip) || exit 1',
    '    echo "Xong phần $K"',
    '  fi',
    'done',
    f'if [ ! -d /content/datasets/{MERGED} ]; then python -m sim_engine.learning.merge_datasets --out /content/datasets/{MERGED} --repo-id local/{MERGED} /content/datasets/{BASE_DS} {parts} || exit 1; fi',
    f'if [ -f {FT_OUT}/checkpoints/last/pretrained_model/train_config.json ]; then',
    f'  lerobot-train --config_path={FT_OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true --dataset.root=/content/datasets/{MERGED} || exit 1',
    'else',
    f'  rm -rf {FT_OUT}',
    f'  python -m sim_engine.learning.train sim_engine/learning/configs/act_box4_topwrist_500_dagger1.yaml --policy.path={BASE_OUT}/checkpoints/{BASE_CKPT}/pretrained_model --dataset.repo_id=local/{MERGED} --dataset.root=/content/datasets/{MERGED} --output_dir={FT_OUT} --job_name={NAME} --steps={FT_STEPS} --num_workers=4 || exit 1',
    'fi',
    f'if [ ! -f {VLA}/eval/{NAME}.json ]; then python -m sim_engine.learning.eval_policy --policy {FT_OUT}/checkpoints/last/pretrained_model --limit 100 --name {NAME} --out {VLA}/eval/{NAME}.json --replay {NAME} || exit 1; fi',
    f'mkdir -p {VLA}/replays && cp /content/sim_engine/results/replays/{NAME}.json {VLA}/replays/ 2>/dev/null',
    # chấm MỌI checkpoint (trừ checkpoint cuối, đã chấm ở trên): điểm dao động mạnh giữa các checkpoint (vòng 2: 78 / 85 / ? / 68)
    f'ls {FT_OUT}/checkpoints | grep -E "^[0-9]+$" | sort | head -n -1 | while read k; do [ -f {VLA}/eval/{NAME}_ck$k.json ] || echo $k; done '
    f'| xargs -P 2 -I KK python -m sim_engine.learning.eval_policy --policy {FT_OUT}/checkpoints/KK/pretrained_model --limit 100 --name {NAME}_ckKK --out {VLA}/eval/{NAME}_ckKK.json',
    f'for f in {VLA}/eval/{NAME}*.json; do echo "CK $(basename $f) $(grep -m1 '"all"' $f)"; done',
    'echo ALL_DONE',
]
open('/content/run_dagger.sh', 'w').write('\n'.join(lines) + '\n')
!nohup bash /content/run_dagger.sh > /content/run.log 2>&1 &
print('Đã chạy nền DAgger vòng', ROUND, '| mạng mới sẽ ở', FT_OUT)

In [ ]:
# Tiến độ DAgger: ghi (seed ...: teacher_ok / ai_ok ...), từng phần, gộp, train (step: ...), chấm ([k/100]); xong khi thấy ALL_DONE
!tr '\r' '\n' < /content/run.log | grep -E "seed [0-9]+: |Xong|gộp|^CK |step:|\[[0-9]+/100\]|\"all\"|ALL_DONE|Error|Traceback" | tail -n 6
!cp /content/run.log {VLA}/{NAME}_run.log

## 11. Train lại từ đầu trên dữ liệu một thầy (min-jerk), chạy nền

Các mạng DAgger 1-3 học trên dữ liệu **lẫn hai thầy**: 500 ván gốc ghi bằng `ScriptedPolicyHybrid` bản gốc (đi nhanh), ván DAgger ghi bằng
bản min-jerk (đi chậm, thả thấp, kéo vòng cung). Cùng cảnh mà hai lệnh khác nhau thì ACT học trung bình, mạng chập chờn (3 mạng tốt nhất
cùng ~601/774 nhưng chỉ 455 seed đạt ở cả ba). Ô dưới, tất cả ở nền:

1. **Ghi 500 ván min-jerk** (policy mặc định): 5 phần x 100 ván chạy song song (seed 3000-4999), xong phần nào zip lên Drive phần đó.
2. **Gộp** với DAgger vòng 2-3 (thầy min-jerk đã sửa "hộp đã ở bát"). Bỏ vòng 1 (thầy chưa sửa).
3. **Train từ đầu** 60k bước (`learning/configs/act_box4_topwrist_mj.yaml`), checkpoint mỗi 10k bước.
4. **Chấm mọi checkpoint trên đủ 774 seed** (chấm 100 seed lệch tới ±5 điểm, không đủ để chọn checkpoint), 3 lượt song song.

Bị ngắt: chạy lại ô 1-4 rồi ô dưới. Phần đã có zip thì bỏ qua, train có checkpoint thì chạy tiếp, checkpoint đã chấm thì không chấm lại.

In [ ]:
D, N = '/content/datasets', 'so101_box4_topwrist_mj'
NAME = 'act_box4_topwrist_mj_colab'
OUT = f'{VLA}/outputs/train/{NAME}'
lines = [
    'set -x',
    'export MUJOCO_GL=egl HF_HOME=/content/hf-cache',
    f'cd /content && mkdir -p {D}',
    # 1) ghi 500 ván: mỗi phần một dải seed riêng 400 seed (đủ cho 100 ván thành công); DAgger dùng 6000+, đánh giá dưới 3000
    'rec() {',
    f'  P={D}/{N}_p$1; Z={VLA}/datasets/{N}_p$1.zip',
    '  if [ -f $Z ]; then [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P); return; fi',
    f'  rm -rf $P; python -m sim_engine.learning.record_dataset --episodes 100 --cameras top,wrist --seed-start $((3000 + $1 * 400)) --root $P --repo-id local/{N}_p$1 > /content/rec_p$1.log 2>&1 || return 1',
    '  (cd $P && zip -qr /content/rec_p$1.zip . && cp /content/rec_p$1.zip $Z.tmp && mv $Z.tmp $Z); echo "Xong ghi phần $1"',
    '}',
    'for K in 0 1 2 3 4; do rec $K & done; wait',
    f'for K in 0 1 2 3 4; do [ -f {VLA}/datasets/{N}_p$K.zip ] || exit 1; done',
    # 2) DAgger vòng 2-3, gộp
    f'for Z in {VLA}/datasets/so101_box4_topwrist_dagger2_p[0-9]*.zip {VLA}/datasets/so101_box4_topwrist_dagger3_p[0-9]*.zip; do '
    f'P={D}/$(basename $Z .zip); [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P) || exit 1; done',
    f'[ -d {D}/{N} ] || python -m sim_engine.learning.merge_datasets --out {D}/{N} --repo-id local/{N} {D}/{N}_p[0-9] '
    f'{D}/so101_box4_topwrist_dagger2_p[0-9]* {D}/so101_box4_topwrist_dagger3_p[0-9]* || exit 1',
    # 3) train từ đầu
    f'if [ -f {OUT}/checkpoints/last/pretrained_model/train_config.json ]; then',
    f'  lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true --dataset.root={D}/{N} || exit 1',
    'else',
    f'  rm -rf {OUT}; python -m sim_engine.learning.train sim_engine/learning/configs/act_box4_topwrist_mj.yaml --dataset.root={D}/{N} --output_dir={OUT} --job_name={NAME} || exit 1',
    'fi',
    # 4) chấm mọi checkpoint trên 774 seed
    f'ls {OUT}/checkpoints | grep -E "^[0-9]+$" | while read k; do [ -f {VLA}/eval/full_mj_$k.json ] || echo $k; done '
    f'| xargs -P 3 -I KK sh -c "python -m sim_engine.learning.eval_policy --policy {OUT}/checkpoints/KK/pretrained_model --name mj_KK '
    f'--out {VLA}/eval/full_mj_KK.json > /content/full_mj_KK.log 2>&1"',
    f'for f in {VLA}/eval/full_mj_*.json; do echo "CK $(basename $f) $(grep -m1 \'"all"\' $f)"; done',
    'echo ALL_DONE',
]
open('/content/run_mj.sh', 'w').write('\n'.join(lines) + '\n')
!nohup bash /content/run_mj.sh > /content/run_mj.log 2>&1 &
print('Đã chạy nền | mạng mới sẽ ở', OUT)

In [ ]:
# Tiến độ: từng phần ghi (seed ...), gộp, train (step: ...), chấm 774 seed ([k/774]); xong khi thấy ALL_DONE
!for K in 0 1 2 3 4; do echo "ghi p$K: $(tr '\r' '\n' < /content/rec_p$K.log 2>/dev/null | grep -E "seed [0-9]+:" | tail -n 1)"; done
!tr '\r' '\n' < /content/run_mj.log | grep -E "Xong|step:|^CK |ALL_DONE|Error|Traceback" | tail -n 4
!for f in /content/full_mj_*.log; do [ -f $f ] && echo "$f: $(tr '\r' '\n' < $f | grep -E '\[[0-9]+/774\]|"all"' | tail -n 1)"; done
!cp /content/run_mj.log {VLA}/act_box4_topwrist_mj_run.log

## 12. lr giảm dần + DAgger vòng 4 (mạng gộp lái), chạy nền

Đi tiếp từ chuỗi DAgger tốt nhất (vòng 3 ck 20k), không làm lại từ đầu (mục 11 / sổ tay mục 13: từ đầu chỉ 59 %). Hai việc chạy
song song, rồi một việc sau:

- **A. Thử lr giảm dần:** train tiếp vòng 3 ck 20k thêm 10k step trên đúng dữ liệu cũ, lr cosine 3e-5 -> 0
  (`learning/configs/act_box4_topwrist_ft_cosine.yaml`). Chấm 4 checkpoint trên 774 seed: điểm còn nhảy không, có hơn 601/774 không.
- **B. Ghi DAgger vòng 4:** 10 phần x 50 ván, 3 phần chạy cùng lúc; AI lái = **gộp 3 checkpoint** (84,8 %), thầy min-jerk sửa.
- **C. (sau A và B) Train vòng 4:** từ vòng 3 ck 20k, dữ liệu gốc + DAgger 1-4, 20k step lr cosine; chấm mọi checkpoint trên 774 seed.

Bị ngắt: chạy lại ô 1-4 rồi ô dưới. Phần DAgger đã có zip thì bỏ qua, train có checkpoint thì chạy tiếp, đã chấm thì không chấm lại.

In [ ]:
D = '/content/datasets'
T = f'{VLA}/outputs/train'
BASE = f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model'            # điểm xuất phát của A và C
ENS = ' '.join([f'{T}/act_box4_topwrist_500_dagger2_colab/checkpoints/010000/pretrained_model',   # 3 mạng lái chung khi ghi DAgger
                f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/020000/pretrained_model',
                f'{T}/act_box4_topwrist_500_dagger3_colab/checkpoints/030000/pretrained_model'])
DAG = lambda r: f'so101_box4_topwrist_dagger{r}'
M3, M4 = 'so101_box4_topwrist_500_dagger3', 'so101_box4_topwrist_500_dagger4'
OUT_A, OUT_C = f'{T}/act_box4_topwrist_d3cos_colab', f'{T}/act_box4_topwrist_500_dagger4_colab'
CFG = 'sim_engine/learning/configs/act_box4_topwrist_ft_cosine.yaml'
unzip_parts = lambda r: (f'for Z in {VLA}/datasets/{DAG(r)}_p[0-9]*.zip; do P={D}/$(basename $Z .zip); '
                         f'[ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P) || exit 1; done')
parts = lambda rs: ' '.join(f'{D}/{DAG(r)}_p[0-9]*' for r in rs)

def train(out, ds, steps, save):   # train tiếp từ BASE, lr cosine; có checkpoint thì resume
    return [f'if [ -f {out}/checkpoints/last/pretrained_model/train_config.json ]; then',
            f'  lerobot-train --config_path={out}/checkpoints/last/pretrained_model/train_config.json --resume=true --dataset.root={D}/{ds} || exit 1',
            'else',
            f'  rm -rf {out}; python -m sim_engine.learning.train {CFG} --policy.path={BASE} --dataset.repo_id=local/{ds} '
            f'--dataset.root={D}/{ds} --output_dir={out} --job_name=$(basename {out}) --steps={steps} --save_freq={save} || exit 1',
            'fi']

def score(out, tag, par):          # chấm mọi checkpoint chưa chấm trên 774 seed
    return [f'ls {out}/checkpoints | grep -E "^[0-9]+$" | while read k; do [ -f {VLA}/eval/full_{tag}_$k.json ] || echo $k; done '
            f'| xargs -P {par} -I KK sh -c "python -m sim_engine.learning.eval_policy --policy {out}/checkpoints/KK/pretrained_model '
            f'--name {tag}_KK --out {VLA}/eval/full_{tag}_KK.json > /content/full_{tag}_KK.log 2>&1"']

# Một phần DAgger vòng 4 (gọi song song bằng xargs): có zip trên Drive thì chỉ giải nén, chưa có thì ghi rồi zip lên Drive.
part = [
    'K=$1; P=' + D + '/' + DAG(4) + '_p$K; Z=' + VLA + '/datasets/' + DAG(4) + '_p$K.zip',
    'if [ -f $Z ]; then [ -d $P ] || (mkdir -p $P && unzip -q $Z -d $P); exit 0; fi',
    'rm -rf $P',
    f'python -m sim_engine.learning.dagger --policy {ENS} --episodes 50 --seed-start $((36000 + K * 1000)) --root $P '
    f'--repo-id local/{DAG(4)}_p$K > /content/dag4_p$K.log 2>&1 || exit 1',
    '(cd $P && zip -qr /content/dag4_p$K.zip . && cp /content/dag4_p$K.zip $Z.tmp && mv $Z.tmp $Z && rm /content/dag4_p$K.zip) || exit 1',
    'echo "Xong phần $K"',
]
lines = [
    'set -x',
    'export MUJOCO_GL=egl HF_HOME=/content/hf-cache',
    f'cd /content && mkdir -p {D}',
    # dữ liệu vòng 3 (gốc + DAgger 1-3) cho A
    f'[ -d {D}/so101_box4_topwrist_500 ] || (mkdir -p {D}/so101_box4_topwrist_500 && unzip -q {VLA}/datasets/so101_box4_topwrist_500.zip -d {D}/so101_box4_topwrist_500)',
    *[unzip_parts(r) for r in (1, 2, 3)],
    f'[ -d {D}/{M3} ] || python -m sim_engine.learning.merge_datasets --out {D}/{M3} --repo-id local/{M3} {D}/so101_box4_topwrist_500 {parts((1, 2, 3))} || exit 1',
    # B chạy nền: 10 phần, 3 phần cùng lúc
    f'(seq 0 9 | xargs -P 3 -I KK bash /content/dag4_part.sh KK) > /content/dag4.log 2>&1 &',
    'BPID=$!',
    # A: train + chấm
    *train(OUT_A, M3, 10000, 2500),
    *score(OUT_A, 'd3cos', 2),
    'wait $BPID',
    f'for K in $(seq 0 9); do [ -f {VLA}/datasets/{DAG(4)}_p$K.zip ] || exit 1; done',
    # C: gộp gốc + DAgger 1-4, train + chấm
    unzip_parts(4),
    f'[ -d {D}/{M4} ] || python -m sim_engine.learning.merge_datasets --out {D}/{M4} --repo-id local/{M4} {D}/so101_box4_topwrist_500 {parts((1, 2, 3, 4))} || exit 1',
    *train(OUT_C, M4, 20000, 5000),
    *score(OUT_C, 'd4', 3),
    f'for f in {VLA}/eval/full_d3cos_*.json {VLA}/eval/full_d4_*.json; do echo "CK $(basename $f) $(grep -m1 \'"all"\' $f)"; done',
    'echo ALL_DONE',
]
open('/content/dag4_part.sh', 'w').write('\n'.join(part) + '\n')
open('/content/run_d4.sh', 'w').write('\n'.join(lines) + '\n')
!nohup bash /content/run_d4.sh > /content/run_d4.log 2>&1 &
print('Đã chạy nền | A:', OUT_A, '| C:', OUT_C)

In [ ]:
# Tiến độ: DAgger vòng 4 (từng phần), train (step: ...), chấm 774 seed ([k/774]); xong khi thấy ALL_DONE
!for K in $(seq 0 9); do [ -f /content/dag4_p$K.log ] && echo "dagger p$K: $(tr '\r' '\n' < /content/dag4_p$K.log | grep -E "^seed" | tail -n 1 | cut -c1-110)"; done
!tr '\r' '\n' < /content/run_d4.log | grep -E "step:|^CK |ALL_DONE|Error|Traceback" | tail -n 3 | cut -c1-160
!for f in /content/full_d3cos_*.log /content/full_d4_*.log; do [ -f $f ] && echo "$f: $(tail -c 80 $f | tr '\n' ' ')"; done
!cp /content/run_d4.log {VLA}/act_box4_topwrist_500_dagger4_run.log

## 8. Đem checkpoint về máy nhà (khi cần)

Checkpoint đã nằm trên Drive: `MyDrive/vla/outputs/train/<EXP>_colab_b<BATCH>/checkpoints/last/pretrained_model/`.
Tải cả thư mục `pretrained_model` về (vd `D:/vla/outputs/train/...`) rồi chạy `eval_policy --policy <thư mục đó>` như bình thường.